# Fase 1 — Entendimento dos Dados

**Dataset:** IEEE-CIS Fraud Detection (Kaggle)  
**Objetivo desta fase:** Entender a estrutura das duas tabelas (`transaction` e `identity`), o significado dos grupos de variáveis, os valores ausentes e o merge entre as tabelas.

---

## Contexto do Problema

Os dados foram fornecidos pela **Vesta Corporation**, empresa de prevenção de fraudes. Cada linha representa uma **transação financeira** (geralmente um pagamento por e-commerce). O objetivo é prever se uma transação é fraudulenta (`isFraud = 1`) ou legítima (`isFraud = 0`).

### As duas tabelas

| Tabela | Linhas | Colunas | Conteúdo |
|--------|--------|---------|----------|
| `train_transaction` | ~590k | 394 | Dados financeiros da transação |
| `train_identity` | ~141k | 41 | Dados de identidade do dispositivo/usuário |

Nem toda transação tem dados de identidade — apenas ~24% das linhas de `transaction` têm uma linha correspondente em `identity`. Faremos um **LEFT JOIN** para preservar todas as transações.

### Por que LEFT JOIN e não INNER JOIN?

- **INNER JOIN** manteria apenas as ~141k transações que têm identidade → perderíamos ~450k transações, inclusive fraudes
- **LEFT JOIN** mantém todas as ~590k transações, preenchendo com `NaN` onde não há dados de identidade
- No pré-processamento (Fase 3), trataremos esses `NaN` adequadamente

## 1. Imports e Configuração

In [1]:
import os
import sys
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Adiciona o diretório raiz ao path para importar src/utils.py
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_data, merge_tables, missing_summary

# Caminho para os dados brutos (ajuste se necessário)
DATA_DIR = '/home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection'

print(f'Pandas version : {pd.__version__}')
print(f'NumPy version  : {np.__version__}')
print(f'Data dir       : {DATA_DIR}')

Pandas version : 2.2.3
NumPy version  : 1.26.4
Data dir       : /home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection


## 2. Carregamento dos Dados

Os arquivos CSV somam ~1.3 GB no disco. O carregamento pode levar 30–60 segundos dependendo da máquina — é normal.

In [2]:
df_trans, df_id = load_data(DATA_DIR)

Carregando transações de: /home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection/train_transaction.csv
  -> 590,540 linhas, 394 colunas
Carregando identidade de: /home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection/train_identity.csv
  -> 144,233 linhas, 41 colunas


## 3. Inspeção Inicial — Tabela de Transações

A tabela de transações é a principal. Vamos inspecionar sua estrutura antes de qualquer outra coisa.

In [3]:
print('=== TABELA DE TRANSAÇÕES ===')
print(f'Linhas: {df_trans.shape[0]:,}')
print(f'Colunas: {df_trans.shape[1]}')
print(f'Memória: {df_trans.memory_usage(deep=True).sum() / 1e6:.1f} MB')

=== TABELA DE TRANSAÇÕES ===
Linhas: 590,540
Colunas: 394
Memória: 2202.7 MB


In [4]:
# Primeiras linhas — só para ter um senso visual dos dados
df_trans.head(3)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,...,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5000,W,13926,NaN,150.0000,discover,142.0000,credit,315.0000,87.0000,19.0000,NaN,NaN,NaN,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,...,0.0000,0.0000,117.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0000,W,2755,404.0000,150.0000,mastercard,102.0000,credit,325.0000,87.0000,NaN,NaN,gmail.com,NaN,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0000,W,4663,490.0000,150.0000,visa,166.0000,debit,330.0000,87.0000,287.0000,NaN,outlook.com,NaN,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Grupos de variáveis da tabela de transações

O IEEE-CIS organizou as ~394 colunas em grupos com significados diferentes. Entender esses grupos é fundamental antes de qualquer modelagem:

| Grupo | Colunas | Descrição |
|-------|---------|----------|
| **ID** | `TransactionID` | Chave única da transação |
| **Target** | `isFraud` | 0 = legítimo, 1 = fraude |
| **Tempo** | `TransactionDT` | Segundos desde um ponto de referência (não é timestamp real) |
| **Valor** | `TransactionAmt` | Valor em USD da transação |
| **Produto** | `ProductCD` | Categoria do produto (W, H, C, S, R) |
| **Cartão** | `card1`–`card6` | Informações do cartão (banco emissor, tipo, país etc.) |
| **Endereço** | `addr1`, `addr2` | Endereço de cobrança (código de área, país) |
| **Distância** | `dist1`, `dist2` | Distâncias entre endereços associados à transação |
| **E-mail** | `P_emaildomain`, `R_emaildomain` | Domínio do e-mail do comprador (P) e destinatário (R) |
| **Contagem** | `C1`–`C14` | Contadores de comportamento (ex: quantos cartões o endereço usou) |
| **Delta tempo** | `D1`–`D15` | Dias desde eventos anteriores (ex: última transação com o mesmo cartão) |
| **Match** | `M1`–`M9` | Flags booleanas de correspondência (ex: nome no cartão bate com o cadastro?) |
| **Vesta** | `V1`–`V339` | Features proprietárias da Vesta — significado não divulgado |

> **Nota sobre V1–V339:** A Vesta não divulgou o significado dessas features por questões de propriedade intelectual. Mesmo sem saber o que são, o modelo consegue aprender padrões a partir delas. Durante a interpretabilidade (Fase 5 com SHAP), veremos quais delas mais impactam as previsões.

In [5]:
# Categorizar as colunas por grupo para facilitar análises futuras
col_groups = {
    'id':        ['TransactionID'],
    'target':    ['isFraud'],
    'time':      ['TransactionDT'],
    'amount':    ['TransactionAmt'],
    'product':   ['ProductCD'],
    'card':      [c for c in df_trans.columns if c.startswith('card')],
    'addr':      [c for c in df_trans.columns if c.startswith('addr')],
    'dist':      [c for c in df_trans.columns if c.startswith('dist')],
    'email':     ['P_emaildomain', 'R_emaildomain'],
    'C_count':   [c for c in df_trans.columns if c.startswith('C') and c[1:].isdigit()],
    'D_delta':   [c for c in df_trans.columns if c.startswith('D') and c[1:].isdigit()],
    'M_match':   [c for c in df_trans.columns if c.startswith('M') and c[1:].isdigit()],
    'V_vesta':   [c for c in df_trans.columns if c.startswith('V') and c[1:].isdigit()],
}

print('Contagem de colunas por grupo:')
for group, cols in col_groups.items():
    print(f'  {group:10s}: {len(cols):3d} coluna(s)')

Contagem de colunas por grupo:
  id        :   1 coluna(s)
  target    :   1 coluna(s)
  time      :   1 coluna(s)
  amount    :   1 coluna(s)
  product   :   1 coluna(s)
  card      :   6 coluna(s)
  addr      :   2 coluna(s)
  dist      :   2 coluna(s)
  email     :   2 coluna(s)
  C_count   :  14 coluna(s)
  D_delta   :  15 coluna(s)
  M_match   :   9 coluna(s)
  V_vesta   : 339 coluna(s)


In [6]:
# Tipos de dados por grupo — importante para saber o que precisará de encoding
print('Tipos de dados (por grupo):\n')
for group, cols in col_groups.items():
    dtypes = df_trans[cols].dtypes.value_counts()
    print(f'  {group:10s}: {dict(dtypes)}')

Tipos de dados (por grupo):

  id        : {dtype('int64'): 1}
  target    : {dtype('int64'): 1}
  time      : {dtype('int64'): 1}
  amount    : {dtype('float64'): 1}
  product   : {dtype('O'): 1}
  card      : {dtype('float64'): 3, dtype('O'): 2, dtype('int64'): 1}
  addr      : {dtype('float64'): 2}
  dist      : {dtype('float64'): 2}
  email     : {dtype('O'): 2}
  C_count   : {dtype('float64'): 14}
  D_delta   : {dtype('float64'): 15}
  M_match   : {dtype('O'): 9}
  V_vesta   : {dtype('float64'): 339}


## 4. Inspeção Inicial — Tabela de Identidade

A tabela de identidade contém informações sobre o dispositivo e a rede usados na transação.

In [7]:
print('=== TABELA DE IDENTIDADE ===')
print(f'Linhas: {df_id.shape[0]:,}')
print(f'Colunas: {df_id.shape[1]}')
print(f'Memória: {df_id.memory_usage(deep=True).sum() / 1e6:.1f} MB')

df_id.head(3)

=== TABELA DE IDENTIDADE ===
Linhas: 144,233
Colunas: 41
Memória: 165.3 MB


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0000,70787.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0000,NotFound,NaN,-480.0000,New,NotFound,166.0000,NaN,542.0000,144.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0000,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0000,98945.0000,NaN,NaN,0.0000,-5.0000,NaN,NaN,NaN,NaN,100.0000,NotFound,49.0000,-300.0000,New,NotFound,166.0000,NaN,621.0000,500.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0000,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0000,191631.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,0.0000,0.0000,100.0000,NotFound,52.0000,NaN,Found,Found,121.0000,NaN,410.0000,142.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows


In [8]:
# Grupos de variáveis da tabela de identidade
id_col_groups = {
    'id_link':   ['TransactionID'],
    'id_num':    [c for c in df_id.columns if c.startswith('id_') and c[3:].isdigit()],
    'device':    ['DeviceType', 'DeviceInfo'],
}

print('Colunas de identidade por grupo:')
for group, cols in id_col_groups.items():
    print(f'  {group:10s}: {len(cols):3d} coluna(s) — {cols[:5]}{"..." if len(cols) > 5 else ""}')

print('\nTipos de dados:')
print(df_id.dtypes.value_counts())

Colunas de identidade por grupo:
  id_link   :   1 coluna(s) — ['TransactionID']
  id_num    :  38 coluna(s) — ['id_01', 'id_02', 'id_03', 'id_04', 'id_05']...
  device    :   2 coluna(s) — ['DeviceType', 'DeviceInfo']

Tipos de dados:
float64    23
object     17
int64       1
Name: count, dtype: int64


### Variáveis de identidade (id_01 a id_38)

Assim como as variáveis V, a maioria das `id_XX` foi anonimizada. O que se sabe:

- `id_01`–`id_11`: Numéricas — capturam informações de rede e comportamento (ex: score de risco de IP, proxy detection)
- `id_12`–`id_38`: Categóricas — capturam informações do dispositivo, sistema operacional, navegador
- `DeviceType`: `'desktop'` ou `'mobile'` — tipo de dispositivo
- `DeviceInfo`: Modelo específico do dispositivo (ex: `'Windows'`, `'iOS Device'`, `'Samsung SM-...'`)

## 5. Variável Target — isFraud

Antes de qualquer outra análise, precisamos entender o que estamos prevendo.

In [9]:
fraud_counts = df_trans['isFraud'].value_counts()
fraud_pct    = df_trans['isFraud'].value_counts(normalize=True) * 100

print('Distribuição da variável target (isFraud):')
print(f'  Legítimas (0): {fraud_counts[0]:>7,} transações  ({fraud_pct[0]:.2f}%)')
print(f'  Fraudes   (1): {fraud_counts[1]:>7,} transações  ({fraud_pct[1]:.2f}%)')
print(f'  Total         : {len(df_trans):>7,} transações')
print(f'\nRazão de desbalanceamento: {fraud_counts[0]/fraud_counts[1]:.1f}:1')
print('(Para cada transação fraudulenta, há ~X legítimas)')

Distribuição da variável target (isFraud):
  Legítimas (0): 569,877 transações  (96.50%)
  Fraudes   (1):  20,663 transações  (3.50%)
  Total         : 590,540 transações

Razão de desbalanceamento: 27.6:1
(Para cada transação fraudulenta, há ~X legítimas)


> **Interpretação:** O desbalanceamento de ~96.5% vs ~3.5% é um problema clássico em fraude. Se um modelo simplesmente chutasse "legítimo" para tudo, teria 96.5% de acurácia — mas seria inútil. Por isso, na Fase 4 usaremos métricas como **AUC-ROC** e **Precision-Recall**, e técnicas como `class_weight='balanced'` ou SMOTE na Fase 3.

## 6. Análise de Valores Ausentes

Valores ausentes (`NaN`) são muito comuns neste dataset — especialmente nas variáveis V e nas variáveis de identidade. Entender o padrão de missing é crucial para decidir como tratá-los na Fase 3.

In [10]:
# Resumo de missings da tabela de transações
missing_trans = missing_summary(df_trans)

print(f'Colunas com valores ausentes: {len(missing_trans)} de {df_trans.shape[1]}')
print('\nTop 20 colunas com mais missing:')
missing_trans.head(20)

Colunas com valores ausentes: 374 de 394

Top 20 colunas com mais missing:


,missing_count,missing_pct
dist2,552913,93.6%
D7,551623,93.4%
D13,528588,89.5%
D14,528353,89.5%
D12,525823,89.0%
D6,517353,87.6%
D9,515614,87.3%
D8,515614,87.3%
V157,508595,86.1%
V163,508595,86.1%


In [11]:
# Distribuição do % de missing por faixas
missing_pct_numeric = df_trans.isnull().mean() * 100

faixas = {
    'Sem missing (0%)':        (missing_pct_numeric == 0).sum(),
    'Baixo missing (0–25%)':   ((missing_pct_numeric > 0) & (missing_pct_numeric <= 25)).sum(),
    'Médio missing (25–50%)':  ((missing_pct_numeric > 25) & (missing_pct_numeric <= 50)).sum(),
    'Alto missing (50–75%)':   ((missing_pct_numeric > 50) & (missing_pct_numeric <= 75)).sum(),
    'Muito alto missing (>75%)': (missing_pct_numeric > 75).sum(),
}

print('Distribuição de missing na tabela de transações:')
for faixa, count in faixas.items():
    bar = '█' * (count // 5)
    print(f'  {faixa:<30s}: {count:3d} colunas  {bar}')

Distribuição de missing na tabela de transações:
  Sem missing (0%)              :  20 colunas  ████
  Baixo missing (0–25%)         : 162 colunas  ████████████████████████████████
  Médio missing (25–50%)        :  38 colunas  ███████
  Alto missing (50–75%)         :   6 colunas  █
  Muito alto missing (>75%)     : 168 colunas  █████████████████████████████████


In [12]:
# Missings da tabela de identidade
missing_id = missing_summary(df_id)

print(f'Colunas com valores ausentes: {len(missing_id)} de {df_id.shape[1]}')
print('\nTop 15 colunas com mais missing na tabela de identidade:')
missing_id.head(15)

Colunas com valores ausentes: 38 de 41

Top 15 colunas com mais missing na tabela de identidade:


,missing_count,missing_pct
id_24,139486,96.7%
id_25,139101,96.4%
id_07,139078,96.4%
id_08,139078,96.4%
id_21,139074,96.4%
id_26,139070,96.4%
id_27,139064,96.4%
id_23,139064,96.4%
id_22,139064,96.4%
id_18,99120,68.7%


## 7. Estatísticas Descritivas das Variáveis Principais

Antes do merge, vamos inspecionar as variáveis mais interpretáveis.

In [13]:
# TransactionDT — é em segundos desde uma referência, não um timestamp real
print('TransactionDT (segundos desde referência):')
print(f'  Mínimo: {df_trans["TransactionDT"].min():,}')
print(f'  Máximo: {df_trans["TransactionDT"].max():,}')
print(f'  Span em dias: {(df_trans["TransactionDT"].max() - df_trans["TransactionDT"].min()) / 86400:.1f}')

TransactionDT (segundos desde referência):
  Mínimo: 86,400
  Máximo: 15,811,131
  Span em dias: 182.0


In [14]:
# TransactionAmt — valor da transação em USD
print('TransactionAmt (USD):')
print(df_trans['TransactionAmt'].describe().to_string())
print(f'\nTransações acima de $1000: {(df_trans["TransactionAmt"] > 1000).sum():,} ({(df_trans["TransactionAmt"] > 1000).mean()*100:.1f}%)')

TransactionAmt (USD):
count   590540.0000
mean       135.0272
std        239.1625
min          0.2510
25%         43.3210
50%         68.7690
75%        125.0000
max      31937.3910

Transações acima de $1000: 7,267 (1.2%)


In [15]:
# ProductCD — categoria do produto
print('ProductCD (categoria do produto):')
print(df_trans['ProductCD'].value_counts().to_string())
print('\nTaxa de fraude por ProductCD:')
print(df_trans.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False).map('{:.2%}'.format).to_string())

ProductCD (categoria do produto):
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628

Taxa de fraude por ProductCD:
ProductCD
C    11.69%
S     5.90%
H     4.77%
R     3.78%
W     2.04%


In [16]:
# card4 e card6 — geralmente são tipo de cartão (Visa/Mastercard) e tipo (crédito/débito)
print('card4:')
print(df_trans['card4'].value_counts().head(10).to_string())
print('\ncard6:')
print(df_trans['card6'].value_counts().head(10).to_string())

card4:
card4
visa                384767
mastercard          189217
american express      8328
discover              6651

card6:
card6
debit              439938
credit             148986
debit or credit        30
charge card            15


In [17]:
# P_emaildomain — domínio do e-mail do comprador
print('Top 15 domínios de e-mail do comprador (P_emaildomain):')
print(df_trans['P_emaildomain'].value_counts().head(15).to_string())

print('\nTaxa de fraude por domínio (top 10 por volume):')
top_domains = df_trans['P_emaildomain'].value_counts().head(10).index
fraud_by_domain = df_trans[df_trans['P_emaildomain'].isin(top_domains)].groupby('P_emaildomain')['isFraud'].mean()
print(fraud_by_domain.sort_values(ascending=False).map('{:.2%}'.format).to_string())

Top 15 domínios de e-mail do comprador (P_emaildomain):
P_emaildomain
gmail.com        228355
yahoo.com        100934
hotmail.com       45250
anonymous.com     36998
aol.com           28289
comcast.net        7888
icloud.com         6267
outlook.com        5096
msn.com            4092
att.net            4033
live.com           3041
sbcglobal.net      2970
verizon.net        2705
ymail.com          2396
bellsouth.net      1909

Taxa de fraude por domínio (top 10 por volume):
P_emaildomain
outlook.com      9.46%
hotmail.com      5.30%
gmail.com        4.35%
icloud.com       3.14%
comcast.net      3.12%
anonymous.com    2.32%
yahoo.com        2.28%
msn.com          2.20%
aol.com          2.18%
att.net          0.74%


## 8. Merge das Tabelas

Agora fazemos o LEFT JOIN para criar o dataset completo. O campo de junção é `TransactionID`.

In [18]:
df = merge_tables(df_trans, df_id)

# Verificação de integridade após o merge
print(f'\nTransações com dados de identidade: {df["DeviceType"].notna().sum():,} ({df["DeviceType"].notna().mean()*100:.1f}%)')
print(f'Transações sem dados de identidade: {df["DeviceType"].isna().sum():,} ({df["DeviceType"].isna().mean()*100:.1f}%)')

# Verificar se o target foi preservado
assert df['isFraud'].notna().all(), 'ERRO: isFraud tem NaN após o merge!'
assert len(df) == len(df_trans), 'ERRO: número de linhas mudou após o merge!'
print('\n✓ Integridade verificada: todas as transações preservadas, isFraud sem NaN')

Dataset após merge: 590,540 linhas, 434 colunas

Transações com dados de identidade: 140,810 (23.8%)
Transações sem dados de identidade: 449,730 (76.2%)

✓ Integridade verificada: todas as transações preservadas, isFraud sem NaN


In [19]:
# Será que fraudes têm mais dados de identidade?
# (fraudes podem ocorrer com mais frequência de dispositivos conhecidos/rastreáveis)
print('% de transações COM dados de identidade, por classe:')
has_identity = df['DeviceType'].notna()
print(df.groupby('isFraud')[has_identity.name].apply(
    lambda x: f'{has_identity[x.index].mean()*100:.1f}%'
))

# Forma mais clara
for label, name in [(0, 'Legítimas'), (1, 'Fraudes')]:
    subset = has_identity[df['isFraud'] == label]
    print(f'  {name}: {subset.mean()*100:.1f}% têm dados de identidade')

% de transações COM dados de identidade, por classe:
isFraud
0    22.7%
1    54.3%
Name: DeviceType, dtype: object
  Legítimas: 22.7% têm dados de identidade
  Fraudes: 54.3% têm dados de identidade


## 9. Visão Geral do Dataset Final

Após o merge, temos nosso dataset de trabalho para as próximas fases.

In [20]:
print('=== DATASET FINAL (após merge) ===')
print(f'Linhas  : {df.shape[0]:,}')
print(f'Colunas : {df.shape[1]}')
print(f'Memória : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print()

# Tipos de dados no dataset final
print('Tipos de dados:')
print(df.dtypes.value_counts().to_string())
print()

# Total de colunas numéricas vs categóricas
num_cols  = df.select_dtypes(include='number').columns.tolist()
cat_cols  = df.select_dtypes(include='object').columns.tolist()
print(f'Colunas numéricas  : {len(num_cols)}')
print(f'Colunas categóricas: {len(cat_cols)}')
print(f'Categóricas: {cat_cols}')

=== DATASET FINAL (após merge) ===
Linhas  : 590,540
Colunas : 434
Memória : 2.69 GB

Tipos de dados:
float64    399
object      31
int64        4

Colunas numéricas  : 403
Colunas categóricas: 31
Categóricas: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [21]:
# Salvar o dataset merged para uso nas próximas fases
# Parquet é um formato binário muito mais eficiente que CSV:
# leitura ~10x mais rápida, arquivo ~3x menor, e preserva os tipos de dados
# (evita ter que recarregar e re-fazer o merge a cada notebook)
import pyarrow  # requer: conda install -c conda-forge pyarrow

output_path = os.path.join(DATA_DIR, 'train_merged.parquet')
df.to_parquet(output_path, index=False, engine='pyarrow')

size_mb = os.path.getsize(output_path) / 1e6
print(f'Dataset salvo em : {output_path}')
print(f'Tamanho no disco : {size_mb:.1f} MB')
print(f'(CSV equivalente seria ~{size_mb * 3:.0f}–{size_mb * 5:.0f} MB)')

Dataset salvo em : /home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection/train_merged.parquet
Tamanho no disco : 84.7 MB
(CSV equivalente seria ~254–424 MB)


## 10. Resumo da Fase 1

### O que aprendemos:

| Aspecto | Descoberta |
|---------|------------|
| **Volume** | 590k transações × 434 colunas após merge |
| **Target** | ~3.5% de fraudes — dataset fortemente desbalanceado |
| **Missing** | Grande volume de missing, especialmente em V1–V339 e variáveis de identidade |
| **Identidade** | Apenas ~24% das transações têm dados de identidade |
| **Variáveis** | Mix de numéricas (maioria) e categóricas (ProductCD, card4, card6, emails, device) |
| **Tempo** | ~6 meses de dados (TransactionDT em segundos) |

### O que faremos a seguir (Fase 2 — EDA):

- Visualizar a distribuição do desbalanceamento
- Analisar distribuições de TransactionAmt por classe (fraude vs legítimo)
- Explorar padrões temporais de fraude
- Identificar as variáveis mais correlacionadas com `isFraud`
- Examinar os missings com mais profundidade (missing é aleatório ou sistemático?)